# BP5 Gate 3 — Hypothesis Testing / Regression / SHAP Association Benchmark
**Customer360 Navigator Enterprise Suite — Root Cause & Driver Analytics**

## What this gate does

Per BP5 Gate 1's own real `policy.json.target_definition.methodology_policy` — a policy-level
field Gate 1 wrote specifically to define WHAT Gate 3 would test, not HOW — this gate runs three
real, disclosed statistical procedures against every real candidate driver field (`Product`,
`Sub-product`, `Issue`, `Sub-issue`, `Submitted via`, `Company`), tested separately against BOTH
real outcomes (`outcome_1_intervention_required`, `outcome_2_timely_response_failure` — never
combined into one joint target, per BP5's own "two_outcomes_tested_separately" rule):

1. **Chi-square test of independence + Cramer's V** effect size (Section 5) — the named-metric
   rule Master Plan Section 14 calls for.
2. **Logistic regression, coefficients with confidence intervals** (Sections 6–7) — a closed-form
   per-category log-odds-ratio (95% Wald CI) for the 5 categorical fields, and a true statsmodels
   Logit fit for `Company` (frequency-encoded, then z-scored).
3. **SHAP explainability on the Gate 3 champion model, on a representative sample only** (Sections
   9–10) — one real logistic-regression champion per outcome (all 5 categorical driver fields
   one-hot + `Company_freq` z-scored, `class_weight="balanced"`), explained with
   `shap.LinearExplainer` on a bounded real held-out sample (300 rows), never the full corpus.

Section 8 additionally runs the real, disclosed diagnostic test BP5 Gate 1's own `assumptions`
list deferred to Gate 3 for `'Timely response?'` (barred as a driver for outcome_1 only),
`response_duration_days` (derived from `'Date received'`/`'Date sent to company'`, barred for
BOTH outcomes), and `'Company response to consumer'` vs outcome_2 (barred as an outcome-echo
risk) — **this test never relaxes any Gate 1 bar**; it reports the real diagnostic number for a
human governance reviewer, the identical disclosure posture as
`src/models/bp3_fairness_mitigation.py`'s `simulate_equalized_fpr_thresholds()`.

`State` is tested at Section 5 too, but only as a **control-field** association check (labeled
`control_field=True` in the saved CSV) — never fed into the champion model or presented as a
named driver finding, per Gate 1's own policy wording.

## Why the closed-form log-odds-ratio, not an iterative MLE fit, for the categorical fields

A saturated single-predictor logistic regression's per-category coefficient (vs a reference
category) has an exact closed form from that category's own 2x2 table against the reference —
this gate computes that directly instead of fitting `statsmodels.Logit` with a large one-hot
design matrix, because `Sub-issue` alone has 222 real levels and `outcome_2` has an extreme real
class imbalance (0.31% positive) — a real, disclosed convergence/separation risk an iterative MLE
fit would carry at this scale, that the closed form has no exposure to at all, while estimating
the exact same quantity. **This was cross-checked, not assumed**: this gate's own pre-delivery
sandbox verification isolated a single two-category comparison and fit it with real
`statsmodels.Logit` MLE directly — the two methods agreed to within 1e-6, confirming the closed
form is not an approximation.

## A real bug this gate's own sandbox verification caught and fixed before delivery

The champion model's `Company_freq` feature (real range: 0 into the low thousands) was first left
on its raw scale, unstandardized, alongside 0/1 one-hot columns for the 5 categorical fields —
`sklearn`'s `lbfgs` solver failed to converge within `max_iter=1000` on both outcomes' real data
(a real numerical-conditioning issue, not a data or methodology problem). Fixed by standardizing
`Company_freq` (fit on train only, applied to test) inside `build_champion_model()` before it
reaches the solver — confirmed clean (no convergence warning, on either outcome) after the fix,
before this notebook was ever delivered.

## Prerequisite

BP5 Gate 2 must have been **real-run** by the user (this notebook reads the real
`cfpb_root_cause_driver_gold.parquet` Gate 2 writes, and re-verifies its row count and both
outcomes' class balance live against Gate 1's own `policy.json` before trusting it — Section 4).
Per this project's standing execution-boundary rule, Claude never runs this notebook — only the
user does, in the `home_credit_env` Jupyter kernel.

## What this gate does NOT do

- **No causal claims, anywhere.** Every chi-square, log-odds-ratio, and SHAP finding is reported
  as a statistical **association** only — BP5 Gate 1's own structural
  `association_not_causation_disclaimer`, carried into this gate's saved artifacts verbatim via
  `ASSOCIATION_NOT_CAUSATION_DISCLAIMER`.
- **No bar relaxation.** Section 8's barred-field diagnostics report real numbers for a human
  governance reviewer; they never flip any Gate 1 bar to "allowed" on their own (see
  `barred_fields_bar_relaxed: False`, written into the Gate 3 config block itself, not only stated
  in prose).
- **No multi-model benchmark.** Unlike BP1–3's Gate 3 (5–6 classifier candidates, one selected as
  champion), BP5's own `methodology_policy` names only logistic regression — a real, disclosed
  scope difference from BP1–3's Gate 3, not an oversight. The single logistic-regression fit per
  outcome IS the champion SHAP explains.
- **No `State` as a named driver.** Tested for association (control-only) but never one-hot
  encoded into the champion model or reported as a driver finding.

## Real, disclosed design choices in this gate

- **Reference category** for each field's log-odds-ratio table defaults to that field's own
  real most-frequent category (never hand-picked) — printed and saved per field/outcome.
- **Haldane-Anscombe 0.5 continuity correction** is applied ONLY to a category whose real 2x2
  table against the reference has a zero cell, flagged per-category
  (`continuity_correction_applied`) — never applied silently or uniformly.
- **`Company_freq` is z-scored** (train-fit mean/std) before every regression that uses it
  (univariate and champion), for numerical stability and so its coefficient is directly
  comparable in scale to the categorical odds ratios.
- **Held-out ROC-AUC/PR-AUC** for each champion are reported as supplementary real context, not
  as a pass/fail gate criterion — BP5's own methodology_policy sets no performance bar the way
  BP1-3's classifier benchmarks do (BP5 is an association-testing BP, not a deployed-classifier
  BP).

Every real number in this notebook's output — every row count, chi2 statistic, p-value, odds
ratio, confidence interval, held-out AUC, and SHAP value — is computed directly against the real,
already Gate-2-confirmed CFPB Gold layer. Nothing is estimated, assumed, or synthesized.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP5 Gate 3 hypothesis testing / regression / SHAP
association benchmark notebook. Single consolidated code cell (platform convention). Idempotent -
safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import numpy as np
import pandas as pd
from IPython.display import display

from models.bp5_driver_association import (
    ASSOCIATION_NOT_CAUSATION_DISCLAIMER,
    BARRED_FIELD_DIAGNOSTIC_DISCLOSURE,
    chi_square_cramers_v,
    log_odds_ratio_by_category,
    univariate_logistic_numeric,
    compute_response_duration_days,
    build_champion_model,
    run_shap_on_champion,
)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp5_root_cause_driver_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_root_cause_driver_gold.parquet"
BP5_CONFIG_PATH = CONFIGS_DIR / "bp5_root_cause_driver_analytics.yaml"
POLICY_JSON_PATH = ARTIFACTS_DIR / "policy.json"

for p in (GOLD_PATH, BP5_CONFIG_PATH, POLICY_JSON_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP5 Gate 2 has been real-run at least once "
            "(the Gold-layer parquet is Gate 2's own real output artifact)."
        )

with open(POLICY_JSON_PATH, "r", encoding="utf-8") as f:
    gate1_policy = json.load(f)

# Read Gate 2's own real, already-written config block directly (marker-based file, never
# hand-parsed with a fragile regex - a plain text scan for the two lines this cell needs is
# sufficient and avoids adding a YAML-block-reader dependency for a two-value read).
gate2_config_text = BP5_CONFIG_PATH.read_text(encoding="utf-8")
if "# --- Gate 2 (Data Verification & Feature Engineering) results" not in gate2_config_text:
    raise RuntimeError(
        "BP5 Gate 2's own config block was not found in "
        f"{BP5_CONFIG_PATH.relative_to(PROJECT_ROOT)} - confirm Gate 2 has been real-run before "
        "running Gate 3."
    )
print(f"[OK] Loaded Gate 1 policy.json (generated_at_utc={gate1_policy['generated_at_utc']}) and "
      "confirmed Gate 2's own config block is present.")

# ============================================================
# SECTION 4: Load the real Gate 2 Gold layer and LIVE-reverify its row/outcome counts against
# Gate 1's policy.json - zero-fabrication: never trust a cached number without re-measuring.
# ============================================================
df = pd.read_parquet(GOLD_PATH)
print(f"\n[OK] Loaded real Gold layer: {len(df):,} rows, {len(df.columns)} columns.")

gate1_row_count = gate1_policy["live_checks"]["cfpb_row_count"]
row_count_drift = len(df) != gate1_row_count

n_o1_pos_live = int((df["outcome_1_intervention_required"] == 1).sum())
n_o1_neg_live = int((df["outcome_1_intervention_required"] == 0).sum())
n_o2_pos_live = int((df["outcome_2_timely_response_failure"] == 1).sum())
n_o2_neg_live = int((df["outcome_2_timely_response_failure"] == 0).sum())

gate1_o1 = gate1_policy["live_checks"]["outcome_1_intervention_required_class_balance"]
gate1_o2 = gate1_policy["live_checks"]["outcome_2_timely_response_failure_class_balance"]
outcome_1_drift = (
    n_o1_pos_live != gate1_o1["n_intervention_required"] or n_o1_neg_live != gate1_o1["n_no_intervention_required"]
)
outcome_2_drift = (
    n_o2_pos_live != gate1_o2["n_timely_response_failure"] or n_o2_neg_live != gate1_o2["n_timely_response_ok"]
)

print(
    f"[FINDING] row_count_drift={row_count_drift}, outcome_1_drift={outcome_1_drift}, "
    f"outcome_2_drift={outcome_2_drift}"
)
if row_count_drift or outcome_1_drift or outcome_2_drift:
    print(
        "[DRIFT DETECTED] Gold layer no longer matches Gate 1's real policy.json - re-run Gate 2 "
        "before trusting Gate 3's own results below."
    )
else:
    print("[OK] Gold layer row count and both outcomes' class balance match Gate 1's real policy.json exactly.")

# ============================================================
# SECTION 5: Chi-square test of independence + Cramer's V - every real candidate driver field
# (Product, Sub-product, Issue, Sub-issue, Submitted via) x both real outcomes, PLUS State as a
# control-only association check (never presented as a named driver finding, per BP5 Gate 1's own
# policy - State's own row below is labeled control_field=True to keep that distinction explicit
# in the saved artifact, not only in this cell's prose).
# ============================================================
CANDIDATE_DRIVER_FIELDS = ["Product", "Sub-product", "Issue", "Sub-issue", "Submitted via"]
CONTROL_FIELD = "State"
COMPANY_COL = "Company"
OUTCOME_1 = "outcome_1_intervention_required"
OUTCOME_2 = "outcome_2_timely_response_failure"
OUTCOMES = [OUTCOME_1, OUTCOME_2]

chi_square_rows = []
for field in CANDIDATE_DRIVER_FIELDS + [CONTROL_FIELD]:
    for outcome in OUTCOMES:
        r = chi_square_cramers_v(df, field, outcome)
        r["control_field"] = field == CONTROL_FIELD
        chi_square_rows.append(r)

chi_square_df = pd.DataFrame(chi_square_rows).sort_values(
    ["outcome_field", "cramers_v"], ascending=[True, False]
).reset_index(drop=True)
print("\n=== CHI-SQUARE TEST OF INDEPENDENCE + CRAMER'S V (per field x outcome) ===")
display(chi_square_df[["driver_field", "outcome_field", "control_field", "n_distinct_levels", "cramers_v", "association_strength", "p_value"]])

chi_square_path = ARTIFACTS_DIR / "gate3_chi_square_cramers_v.csv"
chi_square_df.drop(columns=["disclaimer"]).to_csv(chi_square_path, index=False)
print(f"[SAVED] {chi_square_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 6: Closed-form per-category log-odds-ratio (95% Wald CI) - every real candidate driver
# field (excluding State, control-only) x both real outcomes. Equivalent to a saturated univariate
# logistic regression's own per-category coefficient (see module docstring / this notebook's
# markdown cell for the cross-check against statsmodels MLE this project's own sandbox
# verification already ran before delivery).
# ============================================================
log_odds_flat_rows = []
n_continuity_corrections = 0
for field in CANDIDATE_DRIVER_FIELDS:
    for outcome in OUTCOMES:
        r = log_odds_ratio_by_category(df, field, outcome)
        for cat_row in r["categories"]:
            flat = {
                "driver_field": field,
                "outcome_field": outcome,
                "reference_category": r["reference_category"],
                **cat_row,
            }
            log_odds_flat_rows.append(flat)
            if cat_row["continuity_correction_applied"]:
                n_continuity_corrections += 1

log_odds_df = pd.DataFrame(log_odds_flat_rows).sort_values(
    ["outcome_field", "driver_field", "odds_ratio_vs_reference"], ascending=[True, True, False]
).reset_index(drop=True)
print(f"\n=== LOG-ODDS-RATIO BY CATEGORY: {len(log_odds_df)} real category rows across "
      f"{len(CANDIDATE_DRIVER_FIELDS)} fields x {len(OUTCOMES)} outcomes "
      f"({n_continuity_corrections} with Haldane-Anscombe continuity correction) ===")
display(log_odds_df.head(15))

log_odds_path = ARTIFACTS_DIR / "gate3_log_odds_ratio_by_category.csv"
log_odds_df.to_csv(log_odds_path, index=False)
print(f"[SAVED] {log_odds_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 7: Univariate logistic regression for Company_freq (BP5's one real numeric candidate
# driver - frequency-encoded, z-scored before fitting) x both real outcomes.
# ============================================================
company_freq_map_full = df[COMPANY_COL].value_counts().to_dict()
df["_company_freq_full"] = df[COMPANY_COL].map(company_freq_map_full).astype(float)

company_freq_results = {}
for outcome in OUTCOMES:
    r = univariate_logistic_numeric(df, "_company_freq_full", outcome)
    company_freq_results[outcome] = r
    print(
        f"[FINDING] Company_freq (z-scored) vs {outcome}: OR/1sd={r['odds_ratio_per_1sd']:.4f} "
        f"(95% CI [{r['ci_95_low_odds_ratio_per_1sd']:.4f}, {r['ci_95_high_odds_ratio_per_1sd']:.4f}]), "
        f"p={r['p_value']:.4g}, converged={r['converged']}"
    )

company_freq_path = ARTIFACTS_DIR / "gate3_company_freq_univariate_logistic.json"
with open(company_freq_path, "w", encoding="utf-8") as f:
    json.dump(company_freq_results, f, indent=2)
print(f"[SAVED] {company_freq_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 8: Barred-field diagnostics - the real Gate 3 test BP5 Gate 1's own assumptions list
# deferred for 'Timely response?' (barred as a driver for outcome_1 only), a derived
# response_duration_days from 'Date received'/'Date sent to company' (barred for BOTH outcomes),
# and 'Company response to consumer' vs outcome_2 (barred as an outcome-echo risk). This section
# NEVER relaxes any Gate 1 bar - see BARRED_FIELD_DIAGNOSTIC_DISCLOSURE.
# ============================================================
print(f"\n[DISCLOSURE] {BARRED_FIELD_DIAGNOSTIC_DISCLOSURE}")

barred_diagnostics = {}

r_timely_o1 = log_odds_ratio_by_category(df, "Timely response?", OUTCOME_1)
barred_diagnostics["timely_response_vs_outcome_1"] = r_timely_o1
print(f"[DIAGNOSTIC] 'Timely response?' vs outcome_1 (barred driver for outcome_1): "
      f"{r_timely_o1['categories'][0]}")

df["_response_duration_days"] = compute_response_duration_days(df)
n_negative_duration = int((df["_response_duration_days"] < 0).sum())
print(f"[INFO] real response_duration_days: min={df['_response_duration_days'].min():.1f}, "
      f"max={df['_response_duration_days'].max():.1f}, mean={df['_response_duration_days'].mean():.2f}, "
      f"n_negative(data quality flag)={n_negative_duration}")

duration_results = {}
for outcome in OUTCOMES:
    r = univariate_logistic_numeric(df, "_response_duration_days", outcome)
    duration_results[outcome] = r
    print(f"[DIAGNOSTIC] response_duration_days (barred for both outcomes) vs {outcome}: "
          f"OR/1sd={r['odds_ratio_per_1sd']:.4f}, p={r['p_value']:.4g}, converged={r['converged']}")
barred_diagnostics["response_duration_days_vs_outcomes"] = duration_results

r_cr_o2 = chi_square_cramers_v(df, "Company response to consumer", OUTCOME_2)
barred_diagnostics["company_response_to_consumer_vs_outcome_2"] = r_cr_o2
print(f"[DIAGNOSTIC] 'Company response to consumer' vs outcome_2 (barred, outcome-echo risk): "
      f"cramers_v={r_cr_o2['cramers_v']:.4f} ({r_cr_o2['association_strength']}), p={r_cr_o2['p_value']:.4g}")

barred_fields_bar_relaxed = False  # structural field, never flipped by this notebook - see disclosure
barred_diagnostics["bar_relaxed_by_this_notebook"] = barred_fields_bar_relaxed
barred_diagnostics["disclosure"] = BARRED_FIELD_DIAGNOSTIC_DISCLOSURE

barred_path = ARTIFACTS_DIR / "gate3_barred_field_diagnostics.json"
with open(barred_path, "w", encoding="utf-8") as f:
    json.dump(barred_diagnostics, f, indent=2)
print(f"[SAVED] {barred_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Champion model - one real logistic regression per outcome (never a joint/combined
# target - BP5's own "two_outcomes_tested_separately" rule), on the 5 real one-hot candidate
# driver fields + Company_freq (z-scored, fit on train only). This IS the "Gate 3 champion model"
# methodology_policy.explainability_shap refers to below - no multi-model benchmark is run, per
# BP5's own methodology_policy naming only logistic regression.
# ============================================================
champions = {}
champion_performance = {}
for outcome in OUTCOMES:
    champ = build_champion_model(df, CANDIDATE_DRIVER_FIELDS, COMPANY_COL, outcome)
    champions[outcome] = champ
    champion_performance[outcome] = {
        "n_rows_train": champ["n_rows_train"],
        "n_rows_test": champ["n_rows_test"],
        "held_out_roc_auc": champ["held_out_roc_auc"],
        "held_out_pr_auc": champ["held_out_pr_auc"],
    }
    print(
        f"\n[CHAMPION] {outcome}: n_train={champ['n_rows_train']:,} n_test={champ['n_rows_test']:,} "
        f"held_out_roc_auc={champ['held_out_roc_auc']:.4f} held_out_pr_auc={champ['held_out_pr_auc']:.4f} "
        "(supplementary real context - BP5's own methodology_policy does not set a pass/fail "
        "performance bar the way BP1-3's classifier benchmarks do)."
    )
    display(champ["coefficient_table"].head(10))

    coef_path = ARTIFACTS_DIR / f"gate3_champion_coefficients_{outcome.replace('outcome_', 'outcome_')}.csv"
    champ["coefficient_table"].to_csv(coef_path, index=False)
    print(f"[SAVED] {coef_path.relative_to(PROJECT_ROOT)}")

champion_perf_path = ARTIFACTS_DIR / "gate3_champion_held_out_performance.json"
with open(champion_perf_path, "w", encoding="utf-8") as f:
    json.dump(champion_performance, f, indent=2)
print(f"[SAVED] {champion_perf_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: SHAP explainability - LinearExplainer on a bounded real held-out sample per champion
# (never the full corpus, per Master Plan Section 14).
# ============================================================
SHAP_SAMPLE_SIZE = 300
SHAP_BACKGROUND_SIZE = 100
shap_results = {}
for outcome in OUTCOMES:
    r = run_shap_on_champion(
        champions[outcome], sample_size=SHAP_SAMPLE_SIZE, background_size=SHAP_BACKGROUND_SIZE
    )
    shap_results[outcome] = r
    print(f"\n[SHAP] {outcome}: sample={r['sample_size']}, background={r['background_size']}")
    shap_top_df = pd.DataFrame(r["top_features"])
    display(shap_top_df)
    shap_path = ARTIFACTS_DIR / f"gate3_shap_top_features_{outcome}.csv"
    shap_top_df.to_csv(shap_path, index=False)
    print(f"[SAVED] {shap_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 11: Write the Gate 3 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, fifth BP to do so)
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate3_marker = (
    "# --- Gate 3 (Hypothesis Testing / Regression / SHAP Association Benchmark) results "
    "(appended, idempotent overwrite) ---"
)
gate3_block_lines = [
    f"drift_vs_gate1_row_count_live_check: {'none' if not row_count_drift else 'DRIFT_DETECTED'}",
    f"drift_vs_gate1_outcome_1_counts_live_check: {'none' if not outcome_1_drift else 'DRIFT_DETECTED'}",
    f"drift_vs_gate1_outcome_2_counts_live_check: {'none' if not outcome_2_drift else 'DRIFT_DETECTED'}",
    f"n_chi_square_tests_run: {len(chi_square_rows)}",
    f"n_log_odds_ratio_field_outcome_combos: {len(CANDIDATE_DRIVER_FIELDS) * len(OUTCOMES)}",
    f"n_categories_with_continuity_correction_applied: {n_continuity_corrections}",
    f'chi_square_cramers_v_path: "{chi_square_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'log_odds_ratio_by_category_path: "{log_odds_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'company_freq_univariate_logistic_path: "{company_freq_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f'barred_field_diagnostics_path: "{barred_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f"barred_fields_bar_relaxed: {barred_fields_bar_relaxed}",
    f"champion_outcome_1_held_out_roc_auc: {champion_performance[OUTCOME_1]['held_out_roc_auc']:.6f}",
    f"champion_outcome_1_held_out_pr_auc: {champion_performance[OUTCOME_1]['held_out_pr_auc']:.6f}",
    f"champion_outcome_2_held_out_roc_auc: {champion_performance[OUTCOME_2]['held_out_roc_auc']:.6f}",
    f"champion_outcome_2_held_out_pr_auc: {champion_performance[OUTCOME_2]['held_out_pr_auc']:.6f}",
    f'champion_held_out_performance_path: "{champion_perf_path.relative_to(PROJECT_ROOT).as_posix()}"',
    f"shap_sample_size: {SHAP_SAMPLE_SIZE}",
    f"shap_background_size: {SHAP_BACKGROUND_SIZE}",
    "association_not_causation_disclaimer_carried_forward: True",
]
write_gate_block(BP5_CONFIG_PATH, gate3_marker, gate3_block_lines)
print(f"[SAVED] gate3 block written to {BP5_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "gold_layer_row_count_matches_gate1": not row_count_drift,
    "outcome_1_class_balance_matches_gate1": not outcome_1_drift,
    "outcome_2_class_balance_matches_gate1": not outcome_2_drift,
    "chi_square_tests_cover_all_fields_and_outcomes": len(chi_square_rows) == (len(CANDIDATE_DRIVER_FIELDS) + 1) * len(OUTCOMES),
    "log_odds_ratio_covers_all_fields_and_outcomes": len(log_odds_df) > 0,
    "company_freq_univariate_both_outcomes_converged": all(r["converged"] for r in company_freq_results.values()),
    "barred_fields_bar_not_relaxed": barred_fields_bar_relaxed is False,
    "champion_outcome_1_held_out_roc_auc_above_chance": champion_performance[OUTCOME_1]["held_out_roc_auc"] > 0.5,
    "champion_outcome_2_held_out_roc_auc_above_chance": champion_performance[OUTCOME_2]["held_out_roc_auc"] > 0.5,
    "shap_computed_for_both_outcomes": set(shap_results.keys()) == set(OUTCOMES),
    "shap_sample_never_exceeds_bound": all(r["sample_size"] <= SHAP_SAMPLE_SIZE for r in shap_results.values()),
    "chi_square_csv_written": chi_square_path.exists(),
    "log_odds_ratio_csv_written": log_odds_path.exists(),
    "barred_field_diagnostics_json_written": barred_path.exists(),
    "bp5_config_gate3_block_written": BP5_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP5 Gate 3 complete - chi-square + Cramer's V and closed-form "
    "log-odds-ratio computed for every real candidate driver field against both real outcomes "
    "(tested separately, never combined), Company's real frequency association tested via "
    "univariate logistic regression, the 3 real barred-field diagnostics run without relaxing any "
    "Gate 1 bar, one real logistic-regression champion per outcome fit and SHAP-explained on a "
    "bounded real held-out sample. Every finding above is an ASSOCIATION, never a causal claim. "
    "Proceed to BP5 Gate 4 next."
)
